# Unpack IDT order spec sheet zips and extract & combine the first page of every spec sheet

For each order `.zip`: unpack it, take **page 1 of every PDF inside**, and join those pages into one PDF named after
the zip. All the combined PDFs go into one folder, which is packed into a `.tar.gz` you can download.
Run this *before* `idt_primer_spec_sheet_ocr.py`.

Upload your order `.zip` files into the same folder as this notebook (or set `zip_dir` below), then run the cells in order.

In [1]:
# Cell 1: one-time install (pypdf is small and pure Python). Safe to re-run.
%pip install -q pypdf

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Cell 2: settings
from pathlib import Path

zip_dir         = Path(".")                      # folder holding the order .zip files (sub-folders are searched too)
work_dir        = Path("order_unpacked")         # each zip is unpacked into its own sub-folder here
combined_dir    = Path("combined_first_pages")   # the ONE folder that ends up holding every combined PDF
archive_path    = Path("combined_first_pages.tar.gz")
combined_name   = "{stem}_first_pages.pdf"       # {stem} = the zip's file name without ".zip". Keep any order/date tag in
                                                 # it: the OCR script's file-name settings, e.g. ('0906_', [9]),
                                                 # match on the PDF's name
delete_unpacked = False                          # True = delete the unpacked folders once the combined PDFs are built

In [3]:
# Cell 3: find and unpack the zip files
import re, shutil, zipfile

def natural_key(text):
    """Sort so that primer_2 comes before primer_10, ignoring case."""
    return [int(t) if t.isdigit() else t.casefold() for t in re.split(r"(\d+)", str(text))]

def inside(path, folder):
    """True if `path` is `folder` or lies underneath it."""
    path, folder = path.resolve(), folder.resolve()
    return path == folder or folder in path.parents

def is_junk(name):
    """Archive entries that are not real files (macOS resource forks and the like)."""
    parts = Path(name).parts
    return not parts or "__MACOSX" in parts or parts[-1].startswith("._") or parts[-1] == ".DS_Store"

zip_files = sorted(
    (p for p in zip_dir.rglob("*")
     if p.is_file() and p.suffix.lower() == ".zip"
     and not any(part.startswith(".") for part in p.relative_to(zip_dir).parts[:-1])   # skips .ipynb_checkpoints
     and not inside(p, work_dir) and not inside(p, combined_dir)),
    key=natural_key)
print(f"{len(zip_files)} zip file(s) found")

if work_dir.exists():
    shutil.rmtree(work_dir)          # start clean so files from an earlier run can't mix in
work_dir.mkdir(parents=True)

unpacked = []                        # (unique name, folder it was unpacked into, source zip)
used_stems = set()
for zp in zip_files:
    stem, n = zp.stem, 2
    while stem in used_stems:        # two zips with the same name in different folders
        stem, n = f"{zp.stem}_{n}", n + 1
    used_stems.add(stem)
    dest = work_dir / stem
    try:
        with zipfile.ZipFile(zp) as zf:
            members = [m for m in zf.infolist() if not m.is_dir() and not is_junk(m.filename)]
            zf.extractall(dest, members=members)
    except (zipfile.BadZipFile, RuntimeError, NotImplementedError, OSError) as exc:   # RuntimeError = password
        print(f"SKIPPED {zp}: {exc}")
        continue
    unpacked.append((stem, dest, zp))
    print(f"unpacked {zp}  ->  {dest}   ({len(members)} files)")
print(f"\n{len(unpacked)} of {len(zip_files)} zip file(s) unpacked")

9 zip file(s) found
unpacked SpecSheets_23440047.zip  ->  order_unpacked/SpecSheets_23440047   (4 files)
unpacked SpecSheets_23442128.zip  ->  order_unpacked/SpecSheets_23442128   (3 files)
unpacked SpecSheets_23505140.zip  ->  order_unpacked/SpecSheets_23505140   (6 files)
unpacked SpecSheets_23550509.zip  ->  order_unpacked/SpecSheets_23550509   (4 files)
unpacked SpecSheets_23667924.zip  ->  order_unpacked/SpecSheets_23667924   (4 files)
unpacked SpecSheets_23815514.zip  ->  order_unpacked/SpecSheets_23815514   (6 files)
unpacked SpecSheets_23835622.zip  ->  order_unpacked/SpecSheets_23835622   (18 files)
unpacked SpecSheets_23923740.zip  ->  order_unpacked/SpecSheets_23923740   (2 files)
unpacked SpecSheets_23926410.zip  ->  order_unpacked/SpecSheets_23926410   (3 files)

9 of 9 zip file(s) unpacked


In [4]:
# Cell 4: first page of every PDF -> one combined PDF per zip, all written into combined_dir
from pypdf import PdfReader, PdfWriter

combined_dir.mkdir(parents=True, exist_ok=True)
for old_pdf in combined_dir.glob("*.pdf"):
    old_pdf.unlink()                 # drop PDFs from an earlier run (anything else in the folder, e.g. CSVs, is left alone)

for stem, folder, zp in unpacked:
    pdfs = sorted((p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() == ".pdf"),
                  key=lambda p: natural_key(p.relative_to(folder)))
    writer, used, skipped = PdfWriter(), [], []
    for pdf in pdfs:
        try:
            reader = PdfReader(pdf)
            if reader.is_encrypted and not reader.decrypt(""):
                raise ValueError("password protected")
            first_page = reader.pages[0]                 # fails if the PDF has no pages
        except Exception as exc:
            skipped.append((pdf.name, str(exc) or type(exc).__name__))
            continue
        writer.add_page(first_page)
        used.append(str(pdf.relative_to(folder)))
        writer.add_outline_item(used[-1], len(used) - 1)  # PDF bookmark = the source file, to trace a page back
    if not used:
        print(f"{zp.name}: no usable PDFs, nothing written")
    else:
        out = combined_dir / combined_name.format(stem=stem)
        with open(out, "wb") as fh:
            writer.write(fh)
        pages = len(PdfReader(out).pages)
        status = "OK" if pages == len(used) else "PAGE COUNT MISMATCH"
        print(f"{out.name}: {pages} pages from {len(pdfs)} PDF(s) in {zp.name}   [{status}]")
    for name, why in skipped:
        print(f"      skipped {name}: {why}")

SpecSheets_23440047_first_pages.pdf: 4 pages from 4 PDF(s) in SpecSheets_23440047.zip   [OK]
SpecSheets_23442128_first_pages.pdf: 3 pages from 3 PDF(s) in SpecSheets_23442128.zip   [OK]
SpecSheets_23505140_first_pages.pdf: 6 pages from 6 PDF(s) in SpecSheets_23505140.zip   [OK]
SpecSheets_23550509_first_pages.pdf: 4 pages from 4 PDF(s) in SpecSheets_23550509.zip   [OK]
SpecSheets_23667924_first_pages.pdf: 4 pages from 4 PDF(s) in SpecSheets_23667924.zip   [OK]
SpecSheets_23815514_first_pages.pdf: 6 pages from 6 PDF(s) in SpecSheets_23815514.zip   [OK]
SpecSheets_23835622_first_pages.pdf: 18 pages from 18 PDF(s) in SpecSheets_23835622.zip   [OK]
SpecSheets_23923740_first_pages.pdf: 2 pages from 2 PDF(s) in SpecSheets_23923740.zip   [OK]
SpecSheets_23926410_first_pages.pdf: 3 pages from 3 PDF(s) in SpecSheets_23926410.zip   [OK]


In [5]:
# Cell 5: pack the combined PDFs into one .tar.gz and show a download link
import tarfile
from IPython.display import FileLink, display

if delete_unpacked and work_dir.exists():
    shutil.rmtree(work_dir)

combined_pdfs = sorted(combined_dir.glob("*.pdf"), key=natural_key)
if not combined_pdfs:
    print("No combined PDFs were made, so there is nothing to archive.")
else:
    with tarfile.open(archive_path, "w:gz") as tar:
        for p in combined_pdfs:                              # only the PDFs; extracts to a folder named like combined_dir
            tar.add(p, arcname=f"{combined_dir.name}/{p.name}")
    print(f"{len(combined_pdfs)} combined PDF(s) packed into {archive_path}  ({archive_path.stat().st_size / 1e6:.1f} MB):")
    for p in combined_pdfs:
        print("   ", p.name)
    display(FileLink(str(archive_path)))   # in JupyterLab: right-click the file in the file browser > Download

9 combined PDF(s) packed into combined_first_pages.tar.gz  (18.2 MB):
    SpecSheets_23440047_first_pages.pdf
    SpecSheets_23442128_first_pages.pdf
    SpecSheets_23505140_first_pages.pdf
    SpecSheets_23550509_first_pages.pdf
    SpecSheets_23667924_first_pages.pdf
    SpecSheets_23815514_first_pages.pdf
    SpecSheets_23835622_first_pages.pdf
    SpecSheets_23923740_first_pages.pdf
    SpecSheets_23926410_first_pages.pdf


/home/jovyan/combined_first_pages.tar.gz

**Put the script `idt_primer_spec_sheet_ocrOct29-2026.py` into the `combined_first_pages` directory!**

In [6]:
%cd combined_first_pages

/home/jovyan/combined_first_pages


In [7]:
%pip install -q pytesseract pdf2image

Note: you may need to restart the kernel to use updated packages.


In a terminal, run the following:

```
conda install -c conda-forge poppler tesseract -y
```

WHEN THAT COMPLETES, YOU CAN RUN...

In [10]:
!mv idt_primer_spec_sheet_ocrOct29-2026.py idt_primer_spec_sheet_ocr.py

In [11]:
pwd

'/home/jovyan/combined_first_pages'

In [ ]:
from pathlib import Path

current_dir = Path(".") 
pdf_files = sorted(
    (p for p in current_dir.rglob("*")
     if p.is_file() and p.suffix.lower() == ".pdf"
     and not any(part.startswith(".") for part in p.relative_to(current_dir).parts[:-1]) )  # skips .ipynb_checkpoints
,key=natural_key)
print(f"{len(pdf_files)} PDF file(s) found")

for pf in pdf_files:
    %run idt_primer_spec_sheet_ocr.py {pf}

9 PDF file(s) found
p1.1: Actin_FP.1 | 20 bases | Tm 56.0 | OK 
p2.1: ACtin_RP.1 | 20 bases | Tm 58.9 | OK 


With the information collected from the sheets, we need to do some housekeeping to prepare for the combining script that boils the collected information down to just what is needed in the primer list.  
We'll create a directory and place any `.csv` files with the extracted data in that directory so that we can subsequently point the combining script at that directory.

In [ ]:
!mkdir -p extracted_csvs

In [6]:
!mv *.csv extracted_csvs/

In [4]:
!curl -OL https://raw.githubusercontent.com/fomightez/idt_spec_sheet_processing/refs/heads/main/scripts/idt_primer_spec_sheets_combine.py

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 27876  100 27876    0     0  91455      0 --:--:-- --:--:-- --:--:-- 91697


Now take the mined data and put it into a 'boiled-down' form that is just the needed information from  the generated `.csv` files.

In [7]:
%run idt_primer_spec_sheets_combine.py extracted_csvs

Searched: /home/jovyan/extracted_csvs
CSV files used: 1   skipped: 0
Rows read: 2   exact duplicates removed (same name and sequence): 0   renamed duplicates merged (same sequence, different name): 0   final entries: 20
Status of final entries: OK=17, OK_LENGTH_ONLY=2, REVIEW=1
WARNING, name overrides that matched no row: '18s rRNAR'

Wrote 2 entries to /home/jovyan/extracted_csvs/combined_primers.tsv


**Don't directly open the produced `.tsv` file with Excel.** You should open it with a text editor and verify it shows the degree symbols correctly and then copy-paste that into an new spreadsheet. Then the degree and other symbols won't come out garbled. (I suspect same holds for other Spreadsheet software, like Google Sheets.)